# Session 3 · Part 3 — Interpret molecular landscapes

**Independent checkpoint:** choose the best-correlated protein, compare transcript/observed/predicted maps, and save interpretation prompts.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import matplotlib.pyplot as plt

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import protein_correlations
from dgat_tutorial.plotting import plot_spatial_feature

dataset = load_tutorial_data(paths.raw_data, allow_demo=False)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
if common_spots.empty:
    raise ValueError("Observed and predicted data have no shared spot IDs.")

spots = dataset.spots.loc[common_spots]
observed = dataset.proteins.loc[common_spots]
predicted = predicted.loc[common_spots]
transcripts = dataset.transcripts.loc[common_spots]
correlations = protein_correlations(observed, predicted)
protein = correlations.iloc[0]["protein"]
gene = transcripts.var(axis=0).sort_values(ascending=False).index[0]
print(f"Comparing transcript {gene} with protein {protein}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_spatial_feature(spots, transcripts[gene], f"Transcript {gene}", cmap="magma", ax=axes[0])
plot_spatial_feature(spots, observed[protein], f"Observed {protein}", ax=axes[1])
plot_spatial_feature(spots, predicted[protein], f"Predicted {protein}", ax=axes[2])
plt.tight_layout()
figure_path = paths.figures / "session03_landscape_comparison.png"
plt.savefig(figure_path, dpi=160)
plt.show()


In [ ]:
prompts = (
    "# Session 3 interpretation prompts\n\n"
    "- Which proteins are predicted well by correlation but less well by spatial coherence?\n"
    "- Which proteins show plausible spatial structure despite moderate pointwise correlation?\n"
    "- Where might transcript-protein discordance be biological rather than model error?\n"
    "- What batch, antibody, tissue-boundary, or cell-composition effects could mislead evaluation?\n"
)
prompt_path = paths.results / "session03_interpretation_prompts.md"
prompt_path.write_text(prompts, encoding="utf-8")
manifest = write_checkpoint(
    "3.3", [figure_path, prompt_path],
    summary={"transcript": gene, "protein": protein}, start=paths.root
)
print(prompts)
print(f"Checkpoint written: {manifest}")


## Checkpoint

The tutorial is complete when you can explain where pointwise accuracy and spatial coherence agree or diverge.